# NB-403: KV Cache Optimization

Implement KV cache for efficient inference.

## Learning Objectives
- KV cache concept
- Cache implementation
- Streaming generation
- Memory optimization

In [ ]:
import torch
import torch.nn as nn

print("Ready for KV cache")

## 1. KV Cache Implementation

In [ ]:
class KVCache:
    def __init__(self, batch_size, num_heads, head_dim, max_len, dtype=torch.float16):
        self.batch_size = batch_size
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.max_len = max_len
        self.dtype = dtype

        # TODO: Pre-allocate cache tensors
        self.key_cache = torch.zeros(
            (batch_size, num_heads, max_len, head_dim),
            dtype=dtype,
            device='cuda' if torch.cuda.is_available() else 'cpu'
        )
        self.value_cache = torch.zeros(
            (batch_size, num_heads, max_len, head_dim),
            dtype=dtype,
            device='cuda' if torch.cuda.is_available() else 'cpu'
        )
        self.current_len = 0

    def update(self, new_keys, new_values):
        """Update cache with new K, V."""
        seq_len = new_keys.size(2)

        # TODO: Write to cache
        start_pos = self.current_len
        end_pos = start_pos + seq_len

        self.key_cache[:, :, start_pos:end_pos, :] = new_keys
        self.value_cache[:, :, start_pos:end_pos, :] = new_values

        self.current_len += seq_len

        # Return cached K, V
        return (
            self.key_cache[:, :, :self.current_len, :],
            self.value_cache[:, :, :self.current_len, :],
        )

    def reset(self):
        """Reset cache."""
        self.current_len = 0
        self.key_cache.zero_()
        self.value_cache.zero_()

# Test
cache = KVCache(batch_size=2, num_heads=4, head_dim=64, max_len=1024)
print(f"Cache initialized: {cache.key_cache.shape}")

## 2. Attention with KV Cache

In [ ]:
class CachedAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.qkv = nn.Linear(d_model, 3 * d_model)

    def forward(self, x, kv_cache=None):
        batch_size, seq_len, _ = x.shape

        # TODO: Project Q, K, V
        qkv = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [t.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2) for t in qkv]

        # TODO: Update cache
        if kv_cache is not None:
            k, v = kv_cache.update(k, v)

        # Compute attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = torch.softmax(scores, dim=-1)
        output = torch.matmul(attn, v)

        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        return output

# Test
attn = CachedAttention(d_model=256, n_heads=4)
x = torch.randn(1, 10, 256)
cache = KVCache(1, 4, 64, 1024)

output = attn(x, cache)
print(f"Output shape: {output.shape}")
print(f"Cache position: {cache.current_len}")

## 3. Streaming Generation

In [ ]:
def generate_with_cache(model, tokenizer, prompt, max_new_tokens=20):
    """Generate text using KV cache."""

    # Process prompt
    inputs = tokenizer(prompt, return_tensors="pt")

    # TODO: Initialize cache
    cache = KVCache(
        batch_size=1,
        num_heads=model.config.num_attention_heads,
        head_dim=model.config.hidden_size // model.config.num_attention_heads,
        max_len=4096
    )

    # Initial forward pass
    with torch.no_grad():
        outputs = model(**inputs, past_key_values=cache, use_cache=True)

    generated = inputs["input_ids"]

    # TODO: Streaming generation
    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(
                generated[:, -1:],
                past_key_values=cache,
                use_cache=True
            )

        next_token = outputs.logits[:, -1:].argmax(dim=-1)
        generated = torch.cat([generated, next_token], dim=1)

        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(generated[0])

print("Streaming generation function defined")

## Exercise

1. Implement PagedAttention
2. Add multi-query attention
3. Benchmark with/without cache